# HIT140 Assessment 2 – Team Defence Performance

**Student role:** Keke – Team Defence Performance  
**Analysis level:** one row per team  
**Primary stage:** group stage only  
**Comparison:** teams that subsequently reached the knockout stage versus teams eliminated in the group stage

This notebook is a reproducible, HD-focused analysis. It begins with the two raw downloaded files, documents all wrangling decisions, creates useful team-level variables, checks assumptions, reports a confidence interval and Welch two-sample t-test, and adds effect-size and sensitivity analyses.

## 1. Analytic question formulation

**Analytic question**

> During the group stage, was mean goals conceded per match different between teams that subsequently reached the knockout stage and teams eliminated in the group stage?

This is a timing-aware question: only defensive performance recorded **before** the knockout rounds is used in the primary outcome. It therefore avoids mixing additional post-qualification matches into the comparison.

**Practical value:** the analysis examines whether early tournament defensive performance is associated with later progression.

**Distinct focus:** this is a team-level defence analysis. It does not use goalkeeper save percentage or player-level goalkeeping outcomes, so it remains distinct from the goalkeeper-performance task.

### Population, sample, variables and hypotheses

- **Target population for this analysis:** team performances in the 2026 FIFA World Cup.
- **Observed sample/census:** all 48 participating teams with complete FIFA and FBref records.
- **Observational unit:** one team.
- **Primary numerical response:** `Group-stage Goals Conceded per Match`.
- **Grouping variable:** `Progression Group` (`Knockout` or `Group-stage Eliminated`).
- **Parameter $\mu_K$:** mean group-stage goals conceded per match for knockout-stage teams.
- **Parameter $\mu_E$:** mean group-stage goals conceded per match for group-stage-eliminated teams.

Two-sided hypotheses, specified before the inferential calculation:

$$H_0: \mu_K - \mu_E = 0$$

$$H_a: \mu_K - \mu_E \neq 0$$

The significance level is $\alpha=0.05$.

## 2. Data acquisition, wrangling and preparation

**Raw sources**

1. FIFA team statistics: https://www.fifa.com/en/tournaments/mens/worldcup/canadamexicousa2026/statistics/team-statistics?group=gct_goalkeeping&stat=goals_conceded
2. FBref Scores & Fixtures: https://fbref.com/en/comps/1/schedule/World-Cup-Scores-and-Fixtures

FIFA provides tournament-level team goals conceded and clean sheets. FBref provides match, round and score information. The sources are independently cleaned and then reconciled by team name.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

RANDOM_SEED = 140
DATA_DIR = Path('DATA')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

FIFA_RAW_FILE = DATA_DIR / 'defence.xlsx'
FBREF_RAW_FILE = DATA_DIR / 'fbref_world_cup_2026_scores_fixtures_raw.txt'
FINAL_DATA_FILE = DATA_DIR / 'keke_defence_final_dataset.xlsx'

### 2.1 Wrangle the copied FIFA table

The downloaded FIFA workbook contains alternating statistics rows and team-name rows because merged website cells were copied into Excel. The code below pairs these rows without modifying the raw file.

In [ ]:
fifa_raw = pd.read_excel(FIFA_RAW_FILE)

fifa_stats = (
    fifa_raw.loc[
        fifa_raw['Team'].isna() & fifa_raw['Goals Conceded'].notna(),
        ['Rank', 'Clean Sheets', 'Goals Conceded']
    ]
    .reset_index(drop=True)
)
fifa_teams = (
    fifa_raw.loc[fifa_raw['Team'].notna(), ['Team']]
    .reset_index(drop=True)
)

assert len(fifa_stats) == len(fifa_teams) == 48
fifa_clean = pd.concat([fifa_teams, fifa_stats], axis=1)
fifa_clean[['Rank', 'Clean Sheets', 'Goals Conceded']] = (
    fifa_clean[['Rank', 'Clean Sheets', 'Goals Conceded']].astype(int)
)
fifa_clean.head()

### 2.2 Wrangle the FBref fixture file

Blank separator rows are removed, country codes are stripped from team labels, a shortened Bosnia and Herzegovina label is standardised, and regulation/extra-time scores are extracted without treating penalty-shootout tallies as match goals.

In [ ]:
fbref_raw = pd.read_csv(FBREF_RAW_FILE)
fbref_matches = fbref_raw.loc[
    fbref_raw['Score'].notna() & fbref_raw['Score'].astype(str).str.strip().ne('')
].copy()

def normalise_team_name(value):
    text = str(value).strip()
    parts = text.split()
    if len(parts) > 1 and len(parts[0]) in (2, 3) and parts[0].islower():
        text = ' '.join(parts[1:])
    parts = text.split()
    if len(parts) > 1 and len(parts[-1]) in (2, 3) and parts[-1].islower():
        text = ' '.join(parts[:-1])
    return {'Bosnia–Herz': 'Bosnia and Herzegovina'}.get(text, text)

def parse_score(value):
    text = str(value)
    while '(' in text and ')' in text:
        left = text.index('(')
        right = text.index(')', left)
        text = text[:left] + text[right + 1:]
    home_goals, away_goals = text.replace('–', '-').split('-')
    return int(home_goals.strip()), int(away_goals.strip())

cleaned_matches = []
for match_number, (_, match) in enumerate(fbref_matches.iterrows(), start=1):
    home_goals, away_goals = parse_score(match['Score'])
    cleaned_matches.append({
        'Match ID': f'M{match_number:03d}',
        'Round': match['Round'],
        'Date': match['Date'],
        'Home Team': normalise_team_name(match['Home']),
        'Home Goals': home_goals,
        'Away Team': normalise_team_name(match['Away']),
        'Away Goals': away_goals,
        'Venue': match['Venue'],
        'Notes': match['Notes'],
    })

matches = pd.DataFrame(cleaned_matches)
print(f'Completed matches retained: {len(matches)}')
matches.head()

### 2.3 Reshape matches into team–match records

Each of the 104 matches is converted into two team-level observations, creating 208 auditable team–match records. This is an intermediate wrangling table only; the t-test later uses one aggregated row per team to avoid pseudoreplication.

In [ ]:
team_match_records = []
for _, match in matches.iterrows():
    team_match_records.extend([
        {
            'Match ID': match['Match ID'],
            'Round': match['Round'],
            'Date': match['Date'],
            'Team': match['Home Team'],
            'Opponent': match['Away Team'],
            'Location': 'Home designation',
            'Goals For': match['Home Goals'],
            'Goals Conceded': match['Away Goals'],
            'Clean Sheet': int(match['Away Goals'] == 0),
        },
        {
            'Match ID': match['Match ID'],
            'Round': match['Round'],
            'Date': match['Date'],
            'Team': match['Away Team'],
            'Opponent': match['Home Team'],
            'Location': 'Away designation',
            'Goals For': match['Away Goals'],
            'Goals Conceded': match['Home Goals'],
            'Clean Sheet': int(match['Home Goals'] == 0),
        },
    ])

team_matches = pd.DataFrame(team_match_records)
assert len(team_matches) == 208
team_matches.head()

### 2.4 Cross-source validation and derived variables

FBref tournament goals conceded are aggregated and compared with FIFA. Qualification is derived from whether a team appears in any non-group round. The primary outcome is then constructed from group-stage records only.

In [ ]:
tournament_summary = (
    team_matches.groupby('Team')
    .agg(
        **{
            'Tournament Matches Played': ('Match ID', 'size'),
            'FBref Tournament Goals Conceded': ('Goals Conceded', 'sum'),
            'Reached Knockout': ('Round', lambda rounds: (rounds != 'Group stage').any()),
        }
    )
    .reset_index()
)

validation = fifa_clean.merge(
    tournament_summary,
    on='Team',
    how='outer',
    indicator=True,
    validate='one_to_one'
)
assert validation['_merge'].eq('both').all()
assert (validation['Goals Conceded'] == validation['FBref Tournament Goals Conceded']).all()
assert validation['Goals Conceded'].sum() == 308

group_stage_team_matches = team_matches.loc[
    team_matches['Round'].eq('Group stage')
].copy()
assert len(group_stage_team_matches) == 144

analysis_df = (
    group_stage_team_matches.groupby('Team')
    .agg(
        **{
            'Group-stage Matches': ('Match ID', 'size'),
            'Group-stage Goals For': ('Goals For', 'sum'),
            'Group-stage Goals Conceded': ('Goals Conceded', 'sum'),
            'Group-stage Clean Sheets': ('Clean Sheet', 'sum'),
        }
    )
    .reset_index()
    .merge(
        validation[[
            'Team', 'Reached Knockout', 'Tournament Matches Played',
            'Goals Conceded', 'Clean Sheets', 'Rank'
        ]],
        on='Team',
        how='inner',
        validate='one_to_one'
    )
)

analysis_df['Progression Group'] = np.where(
    analysis_df['Reached Knockout'],
    'Knockout',
    'Group-stage Eliminated'
)
analysis_df['Group-stage Goals Conceded per Match'] = (
    analysis_df['Group-stage Goals Conceded'] / analysis_df['Group-stage Matches']
)
analysis_df['Group-stage Clean Sheet Rate'] = (
    analysis_df['Group-stage Clean Sheets'] / analysis_df['Group-stage Matches']
)
analysis_df['Group-stage Goal Difference per Match'] = (
    (analysis_df['Group-stage Goals For'] - analysis_df['Group-stage Goals Conceded'])
    / analysis_df['Group-stage Matches']
)

analysis_df = analysis_df.rename(columns={
    'Goals Conceded': 'Tournament Goals Conceded (FIFA)',
    'Clean Sheets': 'Tournament Clean Sheets (FIFA)',
    'Rank': 'FIFA Rank by Goals Conceded',
})
analysis_df = analysis_df[[
    'Team', 'Progression Group', 'Group-stage Matches',
    'Group-stage Goals For', 'Group-stage Goals Conceded',
    'Group-stage Clean Sheets', 'Group-stage Goals Conceded per Match',
    'Group-stage Clean Sheet Rate', 'Group-stage Goal Difference per Match',
    'Tournament Matches Played', 'Tournament Goals Conceded (FIFA)',
    'Tournament Clean Sheets (FIFA)', 'FIFA Rank by Goals Conceded'
]].sort_values('Team').reset_index(drop=True)

assert len(analysis_df) == 48
assert analysis_df['Group-stage Matches'].eq(3).all()
analysis_df.head()

In [ ]:
# Verify that the reproducible Python result matches the supplied final workbook.
saved_final = pd.read_excel(FINAL_DATA_FILE, sheet_name='Final_Data')
pd.testing.assert_frame_equal(
    analysis_df,
    saved_final.sort_values('Team').reset_index(drop=True),
    check_dtype=False,
    check_exact=False,
    atol=1e-10,
)
print('Cross-source and final-workbook validation: PASS')

## 3. Data preparation and sampling

**Eligibility rule:** a team must appear in both raw sources and have three completed group-stage matches. All 48 participating teams met this rule; no cases were excluded and there were no missing primary outcomes.

**Sampling method:** this analysis uses a complete census of the 48 teams in this tournament, not a random subsample. A sampling seed is therefore unnecessary for the primary dataset. The observed teams were not randomly selected from all national teams or all tournaments, so broader generalisation is limited.

**Independence consideration:** the groups are mutually exclusive and each team appears once in the final table. However, teams played one another within the same tournament, so strict independence is imperfect and must be acknowledged.

In [ ]:
sampling_report = pd.Series({
    'Teams in FIFA source': fifa_clean['Team'].nunique(),
    'Teams in FBref source': team_matches['Team'].nunique(),
    'Eligible teams retained': analysis_df['Team'].nunique(),
    'Teams excluded': 48 - analysis_df['Team'].nunique(),
    'Group-stage matches': len(matches.loc[matches['Round'].eq('Group stage')]),
    'Group-stage team-match records': len(group_stage_team_matches),
    'Knockout teams': analysis_df['Progression Group'].eq('Knockout').sum(),
    'Group-stage eliminated teams': analysis_df['Progression Group'].eq('Group-stage Eliminated').sum(),
    'Missing primary outcomes': analysis_df['Group-stage Goals Conceded per Match'].isna().sum(),
})
sampling_report

## 4. Descriptive statistics and confidence intervals

The table reports group size, centre, spread, range, and a 95% t-based confidence interval for each group mean. The primary inferential confidence interval for the difference in means is reported with the t-test below.

In [ ]:
outcome = 'Group-stage Goals Conceded per Match'
group = 'Progression Group'

descriptive = (
    analysis_df.groupby(group)[outcome]
    .agg(n='count', mean='mean', median='median', std='std', minimum='min', maximum='max')
)
descriptive['se'] = descriptive['std'] / np.sqrt(descriptive['n'])
descriptive['ci95_lower'] = (
    descriptive['mean']
    - stats.t.ppf(0.975, descriptive['n'] - 1) * descriptive['se']
)
descriptive['ci95_upper'] = (
    descriptive['mean']
    + stats.t.ppf(0.975, descriptive['n'] - 1) * descriptive['se']
)
descriptive.round(3)

In [ ]:
# Supporting variables add context without replacing the pre-specified primary outcome.
supporting_summary = (
    analysis_df.groupby(group)[[
        'Group-stage Clean Sheet Rate',
        'Group-stage Goal Difference per Match'
    ]]
    .agg(['mean', 'median', 'std'])
)
supporting_summary.round(3)

## 5. Visualisation

The first figure compares distributions and preserves every team observation. The second figure shows which teams contribute to the observed difference. Both figures are saved at 300 dpi for later use in the group presentation.

In [ ]:
group_order = ['Knockout', 'Group-stage Eliminated']
group_values = [
    analysis_df.loc[analysis_df[group].eq(name), outcome].to_numpy()
    for name in group_order
]
rng = np.random.default_rng(RANDOM_SEED)

fig, ax = plt.subplots(figsize=(8.5, 5.5))
box = ax.boxplot(group_values, tick_labels=group_order, patch_artist=True, showmeans=True)
for patch, colour in zip(box['boxes'], ['#7FB3D5', '#F0A35E']):
    patch.set_facecolor(colour)
    patch.set_alpha(0.65)
for position, values in enumerate(group_values, start=1):
    jitter = rng.normal(0, 0.045, size=len(values))
    ax.scatter(np.full(len(values), position) + jitter, values,
               color='#23395B', alpha=0.8, s=34, zorder=3)
ax.set_title('Group-stage Defensive Performance and Later Progression')
ax.set_xlabel('Progression group')
ax.set_ylabel('Group-stage goals conceded per match')
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'Figure_1_Group_Stage_Defence_Boxplot.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plot_df = analysis_df.sort_values(outcome).reset_index(drop=True)
colours = plot_df[group].map({
    'Knockout': '#2166AC',
    'Group-stage Eliminated': '#D95F0E'
})

fig, ax = plt.subplots(figsize=(10, 10.5))
ax.scatter(plot_df[outcome], plot_df['Team'], c=colours, s=48)
ax.axvline(
    analysis_df.loc[analysis_df[group].eq('Knockout'), outcome].mean(),
    color='#2166AC', linestyle='--', linewidth=1.4, label='Knockout mean'
)
ax.axvline(
    analysis_df.loc[analysis_df[group].eq('Group-stage Eliminated'), outcome].mean(),
    color='#D95F0E', linestyle='--', linewidth=1.4, label='Eliminated mean'
)
ax.set_title('Team-level Group-stage Goals Conceded per Match')
ax.set_xlabel('Group-stage goals conceded per match')
ax.set_ylabel('Team')
ax.grid(axis='x', alpha=0.25)
ax.legend(loc='lower right')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'Figure_2_Team_Level_Defence.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Assumption checks

The required t-test is retained, but its conditions are evaluated rather than assumed. The outcome is numerical, groups are mutually exclusive, and the final table contains one row per team. Distribution shape, IQR outliers, normality and variance equality are checked below.

In [ ]:
knockout = analysis_df.loc[analysis_df[group].eq('Knockout'), outcome].dropna()
eliminated = analysis_df.loc[
    analysis_df[group].eq('Group-stage Eliminated'), outcome
].dropna()

assumption_rows = []
tolerance = 1e-12
for group_name, values in [('Knockout', knockout), ('Group-stage Eliminated', eliminated)]:
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    outlier_mask = (
        analysis_df[group].eq(group_name)
        & (
            (analysis_df[outcome] < lower_fence - tolerance)
            | (analysis_df[outcome] > upper_fence + tolerance)
        )
    )
    shapiro = stats.shapiro(values)
    assumption_rows.append({
        'Group': group_name,
        'n': len(values),
        'IQR outliers': int(outlier_mask.sum()),
        'Outlier teams': ', '.join(analysis_df.loc[outlier_mask, 'Team']) or 'None',
        'Shapiro W': shapiro.statistic,
        'Shapiro p-value': shapiro.pvalue,
    })

assumption_table = pd.DataFrame(assumption_rows)
levene = stats.levene(knockout, eliminated, center='median')
print(assumption_table.round(4).to_string(index=False))
print(f"Levene's test: statistic={levene.statistic:.4f}, p-value={levene.pvalue:.6f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (group_name, values) in zip(
    axes,
    [('Knockout', knockout), ('Group-stage Eliminated', eliminated)]
):
    stats.probplot(values, dist='norm', plot=ax)
    ax.set_title(f'Q-Q Plot: {group_name}')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'Figure_3_Normality_QQ_Plots.png', dpi=300, bbox_inches='tight')
plt.show()

**Assumption interpretation**

- The IQR rule identifies no clear outliers after applying a small numerical tolerance at the quartile boundary.
- Shapiro–Wilk gives $p=0.0223$ for knockout teams and $p=0.0863$ for eliminated teams. Approximate normality is therefore questionable in the larger group, which is unsurprising because three group matches create a discrete rate.
- Levene's test gives $p=0.0013$, indicating unequal variances. Welch's test is therefore more appropriate than the pooled equal-variance t-test.
- With group sizes of 32 and 16, Welch's test is still informative, but bootstrap and permutation sensitivity checks are added below because the normality and random-sampling conditions are imperfect.

## 7. Inferential statistics: confidence interval and Welch t-test

The primary effect is defined as $\bar{x}_K-\bar{x}_E$. Negative values mean the knockout group conceded fewer goals per match.

In [ ]:
welch_test = stats.ttest_ind(knockout, eliminated, equal_var=False)
mean_difference = knockout.mean() - eliminated.mean()
variance_knockout = knockout.var(ddof=1)
variance_eliminated = eliminated.var(ddof=1)
se_difference = np.sqrt(
    variance_knockout / len(knockout)
    + variance_eliminated / len(eliminated)
)
welch_df = (
    (variance_knockout / len(knockout) + variance_eliminated / len(eliminated)) ** 2
    / (
        (variance_knockout / len(knockout)) ** 2 / (len(knockout) - 1)
        + (variance_eliminated / len(eliminated)) ** 2 / (len(eliminated) - 1)
    )
)
critical_value = stats.t.ppf(0.975, welch_df)
difference_ci = (
    mean_difference - critical_value * se_difference,
    mean_difference + critical_value * se_difference,
)

primary_results = pd.Series({
    'Knockout mean': knockout.mean(),
    'Group-stage eliminated mean': eliminated.mean(),
    'Mean difference (Knockout - Eliminated)': mean_difference,
    't-statistic': welch_test.statistic,
    'Welch degrees of freedom': welch_df,
    'p-value': welch_test.pvalue,
    '95% CI lower': difference_ci[0],
    '95% CI upper': difference_ci[1],
})
primary_results.round(4)

### Effect size and robustness checks

Hedges' $g$ quantifies the standardised difference with a small-sample correction. A bootstrap confidence interval and a label-permutation test are included as sensitivity checks; these do not replace the required Welch t-test.

In [ ]:
n_knockout = len(knockout)
n_eliminated = len(eliminated)
pooled_sd = np.sqrt(
    ((n_knockout - 1) * variance_knockout + (n_eliminated - 1) * variance_eliminated)
    / (n_knockout + n_eliminated - 2)
)
cohen_d = mean_difference / pooled_sd
small_sample_correction = 1 - 3 / (4 * (n_knockout + n_eliminated) - 9)
hedges_g = small_sample_correction * cohen_d
relative_reduction = (
    (eliminated.mean() - knockout.mean()) / eliminated.mean()
)

rng = np.random.default_rng(RANDOM_SEED)
bootstrap_differences = np.empty(20_000)
for index in range(len(bootstrap_differences)):
    bootstrap_differences[index] = (
        rng.choice(knockout, size=n_knockout, replace=True).mean()
        - rng.choice(eliminated, size=n_eliminated, replace=True).mean()
    )
bootstrap_ci = np.quantile(bootstrap_differences, [0.025, 0.975])

pooled_values = np.concatenate([knockout.to_numpy(), eliminated.to_numpy()])
permutation_differences = np.empty(50_000)
for index in range(len(permutation_differences)):
    permuted = rng.permutation(pooled_values)
    permutation_differences[index] = (
        permuted[:n_knockout].mean() - permuted[n_knockout:].mean()
    )
permutation_p = (
    (np.abs(permutation_differences) >= abs(mean_difference)).sum() + 1
) / (len(permutation_differences) + 1)

robustness_results = pd.Series({
    "Hedges' g": hedges_g,
    'Relative reduction in knockout mean': relative_reduction,
    'Bootstrap 95% CI lower': bootstrap_ci[0],
    'Bootstrap 95% CI upper': bootstrap_ci[1],
    'Permutation p-value': permutation_p,
})
robustness_results.round(4)

## 8. Results and conclusion

Teams that later reached the knockout stage conceded a mean of **1.052 group-stage goals per match**, compared with **2.375** for teams eliminated in the group stage. The estimated mean difference was **-1.323 goals per match**, meaning that the knockout group conceded approximately **55.7% fewer** goals per match relative to the eliminated-group mean.

Welch's two-sample t-test produced $t=-4.487$, approximately $20.893$ degrees of freedom, and $p=0.0002$. The 95% confidence interval for the mean difference was **[-1.936, -0.710]**. Because the p-value is below $\alpha=0.05$ and the interval excludes zero, $H_0$ is rejected.

The standardised difference was large ($g=-1.577$). The bootstrap interval **[-1.885, -0.771]** also excluded zero, and the permutation test gave $p<0.0001$. These checks support the same substantive conclusion: in this dataset, stronger group-stage defensive performance was associated with reaching the knockout stage.

This is evidence of an **association**, not proof that conceding fewer goals independently caused qualification.

## 9. Critical limitations

- The dataset is a complete census of one tournament, not a random sample of all international football teams or tournaments.
- Teams played one another, so strict statistical independence is imperfect.
- Qualification was not randomly assigned and also depends on goals scored, points, group composition and tie-breaking rules.
- The primary rate is based on only three group-stage matches per team, producing a discrete distribution.
- Opponent strength, red cards, tactical choices, extra time and other match context are not controlled.
- The analysis should not be described as causal or automatically generalised to other competitions.

## 10. Reproducibility, documentation and contribution record

**Decision log**

- **2026-09-02 – Data acquisition:** retained the original FIFA workbook and FBref fixture export in the `DATA` folder.
- **2026-09-02 – Wrangling:** standardised team names, handled penalty annotations, created 208 team–match records and validated 308 tournament goals conceded across sources.
- **2026-09-02 – Analytic refinement:** restricted the primary outcome to the group stage to preserve temporal order and avoid adding post-qualification matches to the response.
- **2026-09-02 – Inference:** retained the pre-specified two-sided Welch test and added effect-size, bootstrap and permutation sensitivity checks.

**Keke's contribution:** data acquisition for team defence, reproducible cleaning and validation, construction of team-level defensive variables, descriptive and inferential analysis, visualisation, interpretation and limitation review.

To support the Collaboration and Documentation HD criterion, this notebook, the final dataset, figures and progress notes should be committed to the group's GitHub/shared workspace with clear commit messages. Decisions and progress should also be summarised promptly in Teams. The repository history and team communication—not this notebook alone—provide evidence for that rubric category.

## Presentation-ready story

1. **Problem:** using all tournament matches would mix post-qualification games into the defensive outcome.
2. **Method:** reconstruct 144 group-stage team–match records, aggregate to 48 teams, and compare later progression groups.
3. **Finding:** knockout teams conceded 1.323 fewer goals per group-stage match on average; the 95% CI excluded zero.
4. **Critical interpretation:** the association is large and robust across sensitivity checks, but it is not causal and is limited to this tournament.

This structure is designed to support a logical and critical final presentation rather than presenting disconnected statistics.

## Final checklist

- [x] Distinct, non-trivial analytic question
- [x] Reproducible two-source data acquisition and wrangling
- [x] Data preparation, eligibility rule and sampling method
- [x] Descriptive statistics
- [x] Confidence intervals
- [x] Welch two-sample t-test
- [x] Assumption checks and limitations
- [x] Effect size and robustness checks
- [x] Presentation-ready figures and narrative
- [ ] Upload files and progress evidence to GitHub/shared workspace
- [ ] Post the progress summary and decisions in Teams
- [ ] Integrate the analysis into the group's final presentation